In [ ]:
!pip install "protobuf<=3.20.3"

In [ ]:
import warnings

warnings.filterwarnings("ignore")

import os
import cv2
import random
import pandas as pd
import numpy as np
import shutil
import math
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
import time

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from keras import layers, models
import keras.backend as K
from keras.optimizers import Adam, RMSprop
from keras.layers import Input, Concatenate, ZeroPadding2D, BatchNormalization
from keras.layers import Dense, Dropout, Activation
from keras.layers import Conv2D, MaxPooling2D, AveragePooling2D, GlobalAveragePooling2D
from keras.models import Model
from keras.models import Sequential
from keras.preprocessing import image
from keras.applications import DenseNet121
from keras.applications.densenet import preprocess_input

from sklearn.preprocessing import LabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
from keras.callbacks import ModelCheckpoint
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

print("TF version:", tf.version)
print("Keras version:", tf.keras.version)
print("GPU devices:", tf.config.list_physical_devices("GPU"))


#!pip install pyyaml

In [ ]:
data_path = (
    "/kaggle/input/datasets/dewamardana/dataset-manual-selection/dataset_centralcrop"
)

images = []
labels = []

for subfolder in os.listdir(data_path):

    subfolder_path = os.path.join(data_path, subfolder)
    if not os.path.isdir(subfolder_path):
        continue

    for image_filename in os.listdir(subfolder_path):
        image_path = os.path.join(subfolder_path, image_filename)
        images.append(image_path)

        labels.append(subfolder)

data = pd.DataFrame({"image": images, "label": labels})
data.head()
data.shape

In [ ]:
strat = data["label"]
train_df, dummy_df = train_test_split(
    data, train_size=0.80, shuffle=True, random_state=123, stratify=strat
)

strat = dummy_df["label"]
valid_df, test_df = train_test_split(
    dummy_df, train_size=0.5, shuffle=True, random_state=123, stratify=strat
)

print("Training set shape:", train_df.shape)
print("Validation set shape:", valid_df.shape)
print("Test set shape:", test_df.shape)

In [ ]:
batch_size = 4
img_size = (256, 256)
channels = 3
img_shape = (img_size[0], img_size[1], channels)


tr_gen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode="nearest",
)

ts_gen = ImageDataGenerator()

feat_gen = ImageDataGenerator()

train_gen_noaug = feat_gen.flow_from_dataframe(
    train_df,
    x_col="image",
    y_col="label",
    target_size=img_size,
    class_mode="sparse",
    shuffle=False,
    batch_size=batch_size,
)


train_gen = tr_gen.flow_from_dataframe(
    train_df,
    x_col="image",
    y_col="label",
    target_size=img_size,
    class_mode="categorical",
    color_mode="rgb",
    shuffle=True,
    batch_size=batch_size,
)

valid_gen = ts_gen.flow_from_dataframe(
    valid_df,
    x_col="image",
    y_col="label",
    target_size=img_size,
    class_mode="categorical",
    color_mode="rgb",
    shuffle=False,
    batch_size=batch_size,
)

test_gen = ts_gen.flow_from_dataframe(
    test_df,
    x_col="image",
    y_col="label",
    target_size=img_size,
    class_mode="categorical",
    color_mode="rgb",
    shuffle=False,
    batch_size=batch_size,
)

In [ ]:
# === Konfigurasi dasar ===
img_size = (256, 256)

# === Load DenseNet121 sebagai feature extractor ===
model = DenseNet121(
    weights="imagenet",
    include_top=False,  # hilangkan classifier ImageNet
    pooling="avg",  # GlobalAveragePooling2D otomatis (lebih efisien)
    input_shape=(img_size[0], img_size[1], 3),
)

# Frozen semua bobot DenseNet
for layer in model.layers:
    layer.trainable = False

model.summary(expand_nested=True, line_length=200)

In [ ]:
from sklearn.preprocessing import StandardScaler
import gc

# =====================================================
# ⚙️ KONFIGURASI DASAR
# =====================================================
img_size = (256, 256)
channels = 3

# Ambil output dari layer global average pooling terakhir
feature_model = Model(inputs=model.input, outputs=model.get_layer("avg_pool").output)


# =====================================================
# 🔍 EKSTRAKSI FITUR
# =====================================================
def extract_features(generator, feature_extractor):
    features = feature_extractor.predict(generator, verbose=1)
    labels = np.array(generator.classes)
    return features, labels


# Ekstraksi fitur
feature_start_time = time.time()

print("Ekstraksi fitur training...")
train_features, train_labels = extract_features(train_gen_noaug, feature_model)

feature_end_time = time.time()

print("Ekstraksi fitur testing...")
test_features, test_labels = extract_features(test_gen, feature_model)

feature_extraction_time = feature_end_time - feature_start_time
# # =====================================================
# # 🔹 Encode label ke integer
# # =====================================================
train_labels = train_gen_noaug.classes
valid_labels = valid_gen.classes
test_labels = test_gen.classes


# =====================================================
# ⚖️ NORMALISASI FITUR
# =====================================================
scaler = StandardScaler()
train_features = scaler.fit_transform(train_features)
test_features = scaler.transform(test_features)

# =====================================================
# 🔹 HAPUS OBJEK TIDAK DIPAKAI UNTUK HEMAT MEMORI
# =====================================================
gc.collect()

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Jika ingin GPU:
# from cuml.svm import SVC             # GPU
# Jika CPU:
from sklearn.svm import SVC  # CPU

# ============================================
# 3. Inisialisasi model SVM
# ============================================
svm_model = SVC(kernel="rbf", C=1.0, gamma="scale", probability=True)

print("Training SVM sedang berjalan...\n")

# ============================================
# 4. Training
# ============================================
svm_training_start = time.time()

svm_model.fit(train_features, train_labels)

svm_training_end = time.time()

print("Training selesai!\n")

# =====================================================
# TOTAL SVM TRAINING TIME
# =====================================================
svm_training_time = svm_training_end - svm_training_start

total_training_time = feature_extraction_time + svm_training_time

print(f"SVM Training Time: " f"{svm_training_time:.2f} seconds")


# ============================================
# 5. Prediksi pada data test
# ============================================
svm_predictions = svm_model.predict(test_features)
svm_probabilities = svm_model.predict_proba(test_features)[:, 11]


# ============================================
# 6. Evaluasi
# ============================================
acc = accuracy_score(test_labels, svm_predictions)
print("SVM Model Accuracy: {:.2f}%".format(acc * 100))

print("\n📌 Classification Report:")
print(classification_report(test_labels, svm_predictions))

print("\n📌 Confusion Matrix:")
print(confusion_matrix(test_labels, svm_predictions))

In [ ]:
# =====================================================
# CLASSIFICATION START TIME
# =====================================================
classification_start_time = time.time()

# =====================================================
# SVM PREDICTION
# =====================================================
svm_predictions = svm_model.predict(test_features)

# =====================================================
# CLASSIFICATION END TIME
# =====================================================
classification_end_time = time.time()

# =====================================================
# TOTAL CLASSIFICATION TIME
# =====================================================
total_classification_time = classification_end_time - classification_start_time

print(f"Classification Time: " f"{total_classification_time:.2f} seconds")

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# =====================================================
# PERFORMANCE METRICS
# =====================================================
accuracy = accuracy_score(test_labels, svm_predictions)

precision = precision_score(test_labels, svm_predictions, average="weighted")

recall = recall_score(test_labels, svm_predictions, average="weighted")

f1 = f1_score(test_labels, svm_predictions, average="weighted")

In [ ]:
import joblib

joblib.dump(svm_model, "svm_model.pkl")

model.save("densenet121_imagenet.keras")

# =====================================================
# CNN MODEL SIZE
# =====================================================
cnn_model_size = os.path.getsize("densenet121_imagenet.keras") / (1024 * 1024)

# =====================================================
# SVM MODEL SIZE
# =====================================================
svm_model_size = os.path.getsize("svm_model.pkl") / (1024 * 1024)

# =====================================================
# TOTAL MODEL SIZE
# =====================================================
total_model_size = cnn_model_size + svm_model_size

In [ ]:
cm = confusion_matrix(test_labels, svm_predictions)

plt.figure(figsize=(10, 8))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")

plt.title("Confusion Matrix")

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

print(
    classification_report(
        test_labels, svm_predictions, target_names=list(test_gen.class_indices.keys())
    )
)

In [ ]:
print("\n===================================")
print("FINAL RESULT - SKEMA 3")
print("===================================")

print(f"Accuracy               : {accuracy:.4f}")
print(f"Precision              : {precision:.4f}")
print(f"Recall                 : {recall:.4f}")
print(f"F1-Score               : {f1:.4f}")

print(f"Feature Extraction(s) : " f"{feature_extraction_time:.2f}")

print(f"SVM Training Time(s)  : " f"{svm_training_time:.2f}")

print(f"Total Training Time(s): " f"{total_training_time:.2f}")

print(f"Classification Time(s): " f"{total_classification_time:.2f}")

print(f"CNN Model Size(MB)    : " f"{cnn_model_size:.2f}")

print(f"SVM Model Size(MB)    : " f"{svm_model_size:.2f}")

print(f"Total Model Size(MB)  : " f"{total_model_size:.2f}")

In [ ]:
# from sklearn.model_selection import StratifiedKFold
# from sklearn.preprocessing import StandardScaler
# from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
# from sklearn.svm import SVC
# import numpy as np
# import gc

# # =====================================================
# # ⚙️ PERSIAPAN K-FOLD
# # =====================================================
# X = data["image"].values   # path gambar
# y = data["label"].values   # label

# k = 5
# skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=123)

# all_scores = []
# fold_no = 1

# # =====================================================
# # 🔄 LOOP K-FOLD
# # =====================================================
# for train_index, test_index in skf.split(X, y):

#     print(f"\n==============================")
#     print(f"        FOLD {fold_no}")
#     print(f"==============================")

#     # Buat dataframe train/test per fold
#     train_df = data.iloc[train_index].reset_index(drop=True)
#     test_df  = data.iloc[test_index].reset_index(drop=True)

#     # -------------------------------------------------
#     # 1. Buat generator untuk fold ini
#     # -------------------------------------------------
#     train_gen_fold = feat_gen.flow_from_dataframe(
#         train_df,
#         x_col='image',
#         y_col='label',
#         target_size=img_size,
#         class_mode='sparse',
#         shuffle=False,
#         batch_size=batch_size
#     )

#     test_gen_fold = ts_gen.flow_from_dataframe(
#         test_df,
#         x_col='image',
#         y_col='label',
#         target_size=img_size,
#         class_mode='sparse',
#         shuffle=False,
#         batch_size=batch_size
#     )

#     # -------------------------------------------------
#     # 2. Ekstraksi fitur CNN untuk fold ini
#     # -------------------------------------------------
#     print("Ekstraksi fitur training...")
#     train_features, train_labels = extract_features(train_gen_fold, feature_model)

#     print("Ekstraksi fitur test...")
#     test_features, test_labels = extract_features(test_gen_fold, feature_model)

#     # -------------------------------------------------
#     # 3. Normalisasi
#     # -------------------------------------------------
#     scaler = StandardScaler()
#     train_features = scaler.fit_transform(train_features)
#     test_features  = scaler.transform(test_features)

#     # -------------------------------------------------
#     # 4. Train SVM
#     # -------------------------------------------------
#     svm = SVC(kernel="rbf", C=1.0, gamma="scale")
#     svm.fit(train_features, train_labels)

#     # -------------------------------------------------
#     # 5. Evaluasi
#     # -------------------------------------------------
#     preds = svm.predict(test_features)

#     acc = accuracy_score(test_labels, preds)
#     all_scores.append(acc)

#     print(f"Accuracy Fold {fold_no}: {acc*100:.2f}%\n")
#     print(classification_report(test_labels, preds))
#     print(confusion_matrix(test_labels, preds))

#     fold_no += 1
#     gc.collect()


# # =====================================================
# # 📊 HASIL FINAL
# # =====================================================
# print("\n================================")
# print("        FINAL K-FOLD RESULT")
# print("================================")
# for i, score in enumerate(all_scores, start=1):
#     print(f"Fold {i}: {score*100:.2f}%")

# print("\nAverage Accuracy:", np.mean(all_scores)*100, "%")